# Laboratorio 12.1 — Agentic RAG con LangGraph

Añadiremos reescritura de consultas y grading de evidencia a un flujo RAG, con decisión explícita de reintentar o responder y un límite duro de recuperaciones.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Construir un workflow de reescritura, recuperación y grading de documentos.
- Elegir entre responder, recuperar otra vez o abstenerse según evidencia.
- Limitar iteraciones para impedir loops de recuperación.
- Medir Hit Rate y cobertura de evidencia por separado de la generación.

## 2. Conceptos

Agentic RAG usa pasos de control para revisar si el contexto recuperado basta. Una consulta alternativa puede mejorar recuperación, pero también ampliar el tema y traer ruido. El grader del laboratorio es lexical y transparente; no determina verdad. Si el contexto sigue siendo insuficiente tras el presupuesto, el flujo debe abstenerse.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Pregunta] --> B[Router/reformulador]
    B --> C[Retriever]
    C --> D[Grader de documentos]
    D -->|suficiente| E[Generar respuesta citada]
    D -->|insuficiente y quedan intentos| B
    D -->|limite agotado| F[Abstenerse]
    E --> G[Validador Pydantic]
    F --> G
    G --> H[Respuesta final]
```

## 4. Preparación del entorno

Usamos el Agentic RAG con LangGraph del paquete y el corpus FAQ local. Los nodos aplican reglas y búsqueda local; no hacen llamadas a un LLM ni transfieren documentos.

In [ ]:
import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.agents import run_agentic_rag
from llm_engineering.evaluation.metrics import hit_rate_at_k

raiz_curso = preparar_entorno()
documentos = pd.read_json(raiz_curso / 'data' / 'documents' / 'faq.jsonl', lines=True).to_dict(orient='records')

## 5. Definición del problema

Responderemos preguntas FAQ sobre seguimiento y reembolso. Una consulta puede usar una paráfrasis que no comparte términos con el corpus. Las etiquetas relevantes se declaran antes de evaluar recuperación.

## 6. Línea base

La línea base ejecuta una sola búsqueda sin reformulación. Si no encuentra términos coincidentes, conserva ese fallo en vez de inventar respuesta.

In [ ]:
pregunta_parafraseada = 'quiero que me regresen el dinero'
resultado_base = run_agentic_rag(pregunta_parafraseada, max_retrievals=1)
print(resultado_base['validated_output'])

## 7. Implementación

El workflow reformula después de una búsqueda sin evidencia y usa el resultado del grader para decidir. Inspeccionaremos el historial de consultas, intentos, IDs y puntuación; los documentos siguen siendo contenido no confiable y las citas identifican su origen.

In [ ]:
resultados_agentic = [
    {
        'caso': 'consulta_directa',
        **run_agentic_rag('seguimiento paquete', max_retrievals=2),
        'relevantes': {'envio_seguimiento'},
    },
    {
        'caso': 'parafrasis',
        **run_agentic_rag(pregunta_parafraseada, max_retrievals=2),
        'relevantes': {'devolucion_reembolso', 'devolucion_plazo'},
    },
    {
        'caso': 'sin_evidencia_y_sin_reintento',
        **run_agentic_rag('xyzzy qqqv', max_retrievals=1, force_rag=True),
        'relevantes': set(),
    },
]
filas = []
for caso in resultados_agentic:
    salida = caso['validated_output']
    filas.append({
        'caso': caso['caso'],
        'route': salida['route'],
        'status': salida['status'],
        'generation_mode': salida['generation_mode'],
        'llm_calls': salida['llm_calls'],
        'attempts': salida['retrieval_attempts'],
        'grade_score': salida['grade_score'],
        'source_ids': salida['source_ids'],
        'query_history': caso['query_history'],
        'hit_rate': hit_rate_at_k(salida['source_ids'], caso['relevantes'], max(len(salida['source_ids']), 1)) if caso['relevantes'] else None,
    })
tabla_agentic = pd.DataFrame(filas)
display(tabla_agentic)
tabla_agentic.to_csv(raiz_curso / 'outputs' / '12_01_agentic_rag_runs.csv', index=False)

## 8. Experimentos

Comparamos una y dos recuperaciones para la misma paráfrasis; luego verificamos que una solicitud fuera del dominio vaya a la ruta directa. La generación LLM puede habilitarse sobre una única consulta RAG, con un máximo de una llamada; el modo extractivo local sigue siendo el predeterminado.

In [ ]:
filas_presupuesto = []
for limite in (1, 2):
    resultado = run_agentic_rag(pregunta_parafraseada, max_retrievals=limite)
    salida = resultado['validated_output']
    filas_presupuesto.append({
        'max_retrievals': limite,
        'attempts_used': salida['retrieval_attempts'],
        'status': salida['status'],
        'source_ids': salida['source_ids'],
        'queries': resultado['query_history'],
    })
tabla_presupuestos = pd.DataFrame(filas_presupuesto)
display(tabla_presupuestos)

salida_directa = run_agentic_rag('hola, necesito ayuda')['validated_output']
print('Ruta fuera del dominio RAG:', salida_directa['route'])

EJECUTAR_GENERACION_LLM = False
print('Llamadas máximas de generación si se habilita:', 1)
if EJECUTAR_GENERACION_LLM:
    salida_generada = run_agentic_rag(
        pregunta_parafraseada,
        max_retrievals=2,
        generate_with_llm=True,
        provider='ollama',
    )
    print(salida_generada['validated_output'])

## 9. Evaluación

Hit Rate mide si el conjunto de fuentes contiene al menos un documento relevante; `grade_score` resume cobertura lexical del query reescrito. La abstención es correcta para la consulta sin evidencia, aunque su Hit Rate no aplica cuando no hay documentos relevantes etiquetados.

In [ ]:
metricas_agentic = pd.DataFrame([{
    'consultas': len(tabla_agentic),
    'respuestas_con_evidencia': int((tabla_agentic['status'] == 'answered').sum()),
    'abstenciones': int((tabla_agentic['status'] == 'abstained').sum()),
    'reintentos': int((tabla_agentic['attempts'] > 1).sum()),
    'intentos_maximo_observado': int(tabla_agentic['attempts'].max()),
}])
display(metricas_agentic)
metricas_agentic.to_csv(raiz_curso / 'outputs' / '12_01_agentic_rag_metrics.csv', index=False)

## 10. Discusión

Reescritura puede recuperar sinónimos, pero una expansión genérica puede traer evidencia accidental. El grader lexical también puede sobreestimar suficiencia por una palabra compartida. En producción, usa criterios medibles calibrados con humanos, guarda trazas de cada consulta y mantén el límite como salvaguarda independiente.

## 11. Ejercicios

1. Añade una paráfrasis sobre cargos duplicados y etiqueta documentos antes de probar.
2. Cambia solo el umbral del grader y mide cobertura y falsos positivos.
3. Verifica que una respuesta final cite únicamente IDs recuperados.
4. Añade un límite por tiempo además del límite por número de búsquedas.

## 12. Desafío

Sustituye reescritura determinista por un LLM local, mantén el mismo corpus y etiquetas y limita a dos llamadas de generación y dos recuperaciones. Compara Hit Rate, abstenciones, calidad de respuesta, latencia y costo estimado.

## 13. Ideas clave

- Agentic RAG cierra el bucle entre query, retrieval y grading.
- Reintenta solo con una condición explícita y presupuesto finito.
- Evalúa retrieval antes de atribuir calidad al generador.
- Si la evidencia falta, abstenerse es una salida válida.